In [15]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler,LabelEncoder
from sklearn.impute import SimpleImputer
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split,StratifiedGroupKFold,GridSearchCV
from sklearn.metrics import accuracy_score,f1_score
import joblib

In [16]:
df = pd.read_csv('../data/bank.csv')

In [17]:
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes


In [18]:
df.loc[df.job == 'unknown','job'] = np.nan

In [19]:
df[df.job == 'student']

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
162,28,student,single,secondary,no,949,yes,no,unknown,28,may,1730,1,-1,0,unknown,yes
309,38,student,single,tertiary,no,3316,no,no,unknown,17,jun,1345,3,-1,0,unknown,yes
431,31,student,married,primary,no,46,yes,no,cellular,10,jul,487,1,-1,0,unknown,yes
461,29,student,single,tertiary,no,5,no,no,cellular,15,jul,889,5,-1,0,unknown,yes
480,27,student,married,secondary,no,119,yes,no,cellular,16,jul,781,2,-1,0,unknown,yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10703,28,student,single,tertiary,no,61,yes,no,cellular,16,jul,401,8,-1,0,unknown,no
10826,24,student,single,secondary,no,493,yes,no,cellular,13,may,85,4,358,1,other,no
10867,32,student,single,secondary,no,14,no,no,cellular,19,aug,144,3,136,1,other,no
10957,25,student,single,secondary,no,3090,no,no,cellular,21,aug,139,1,-1,0,unknown,no


In [20]:
df.select_dtypes(include='str').columns

Index(['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact',
       'month', 'poutcome', 'deposit'],
      dtype='str')

In [21]:
x = df.drop(columns=['poutcome','deposit'])
y = df['deposit']

In [22]:
num_cols = x.select_dtypes(include='number').columns
cat_cols = x.select_dtypes(include='str').columns

In [23]:
num_pipeline = Pipeline(
    [
        ('imputer',SimpleImputer(strategy='median')),
        ('scl',StandardScaler())
    ]
)

In [24]:
cat_pipeline = Pipeline(
    [
        ('imputer',SimpleImputer(strategy='most_frequent')),
        ('ohe',OneHotEncoder())
    ]
)

In [25]:
preprocessor = ColumnTransformer(
    [
        ('num',num_pipeline,num_cols),
        ('cat',cat_pipeline,cat_cols)
    ]
)

In [26]:
pipeline = Pipeline(
    [
        ('preprocessor',preprocessor),
        ('model',GradientBoostingClassifier(random_state=42))
    ]
)

In [27]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [28]:
params = {
    'model__loss':['log_loss','exponential'],
    'model__learning_rate':[0.01,0.5,1],
    'model__n_estimators':[100,150,200],
    'model__min_samples_split':[2,5,7],
    'model__max_depth':[3,5,8],
    'model__min_samples_leaf':[1,3,5]
}

In [29]:
grid = GridSearchCV(
    estimator=pipeline,
    param_grid=params,
    cv=4,
    scoring='f1',
    n_jobs=-1
)

In [30]:
grid.fit(x_train,y_train)

/home/anons/dev/ai_learning/practice/venv/lib/python3.12/site-packages/sklearn/model_selection/_validation.py:956: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/home/anons/dev/ai_learning/practice/venv/lib/python3.12/site-packages/sklearn/model_selection/_validation.py", line 945, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/anons/dev/ai_learning/practice/venv/lib/python3.12/site-packages/sklearn/utils/validation.py", line 80, in inner_f
    return f(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^
  File "/home/anons/dev/ai_learning/practice/venv/lib/python3.12/site-packages/sklearn/metrics/_scorer.py", line 322, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__learning_rate': [0.01, 0.5, ...], 'model__loss': ['log_loss', 'exponential'], 'model__max_depth': [3, 5, ...], 'model__min_samples_leaf': [1, 3, ...], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",4
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 S

In [31]:
y_pred = grid.best_estimator_.predict(x_test)

In [32]:
y_pred

array(['no', 'yes', 'yes', ..., 'no', 'yes', 'no'],
      shape=(2233,), dtype=object)

In [33]:
joblib.dump(grid.best_estimator_,'bank_marketing.pkl')

['bank_marketing.pkl']